In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
np.random.seed(42)

# confidence = 0.80       # 0.95 for 95 % Confidence Interval
# test_side = 2           # 1 for one sided, 2 for two sided
# relative_mean = 'Lower' # Claim of relative position of population mean, 'Lower' or 'Higher', needed for 1 sided test

confidence = eval(dbutils.widgets.get("confidence"))
test_side = eval(dbutils.widgets.get("test_side"))
relative_mean = (dbutils.widgets.get("ts_relative_mean"))
alpha = 1 - confidence
prob = 1 - alpha / test_side

def print_stats(test_name, distribution, score, value, lower, upper):
    if distribution == 't':
        print(f"t-score: {score:.5f} for t({prob:.4f}, {df:.2f})")
        print(f"C.I. for {test_name}: ({lower:.5f}, {upper:.5f})")
        print(f"t-value: {value:.5f}")

    if distribution == 'Z':
        print(f"Z-score: {score:.5f} for Z({prob:.4f})")
        print(f"C.I. for {test_name}: ({lower:.5f}, {upper:.5f})")
        print(f"Z-value: {value:.5f}")
    
    print(f"p-value: {p_value* 100:.2f} %")
    if p_value < alpha:
        print("Reject the Null Hypothesis")
    else:
        print("Accept the Null Hypothesis")

def test_output2(distribution, score, value, factor, var):
    if distribution == 't':
      integral = stats.t.cdf(abs(value), df)
    else:
      integral = stats.norm.cdf(abs(value))

    if test_side == 2:
        p_value = 2 * (1 - integral)
        CI_lower =var - score * factor
        CI_upper = var + score * factor
    else:
        if relative_mean == "Lower": 
            p_value = integral
            CI_lower = -np.inf
            CI_upper = var + score * factor
        else:  
            p_value = 1 - integral
            CI_lower = var - score * factor
            CI_upper = np.inf

    return p_value, CI_lower, CI_upper

# Measure Central Tendency

**Mean Median Mode Range IQR Variance Standard Dev**

In [0]:
# Insert Sample Data
data = [10, 20, 50, 40, 50, 60, 70, 80]

In [0]:
# Insert Sample Data
s = pd.Series(data)

# 8. Visualizations
plt.figure(figsize=(6, 3))

# Subplot 1: Histogram
plt.subplot(1, 2, 1)
plt.hist(s, color='skyblue', edgecolor='black', alpha=0.7)
plt.title("Distribution of Data")
plt.xlabel("Value")
plt.ylabel("Frequency")

# Plot 2: Box Plot
plt.subplot(1, 2, 2)
plt.boxplot(s)
plt.title("Box Plot of Data")
plt.ylabel("Values")

plt.tight_layout()
plt.show()

In [0]:
# 1. Mean (Average)
mean_val = s.mean()

# 2. Median (Middle value)
median_val = s.median()

# 3. Mode (Most frequent value)
# Note: Returns a Series (use .iloc[0] for the first mode)
mode_val = s.mode().iloc[0]

# 4. Range (Max - Min)
data_range = s.max() - s.min()

# 5. IQR (Interquartile Range: Q3 - Q1)
iqr_val = s.quantile(0.75) - s.quantile(0.25)

# 6. Variance
variance_val = s.var()

# 7. Standard Deviation
std_dev_val = s.std()

# Print results
print("Sample Statistics:\n")
print(f"Count:{len(s)},  Mean: {mean_val}, Std Dev: {std_dev_val:.5f}, Variance: {variance_val:.5f}")
print(f"Median: {median_val}, Range: {data_range}, IQR: {iqr_val}, Mode: {mode_val}")

# Type of tests

|Type | Condition | Test     | Reasoning           |
|---| ------------------------------------------------------------------------------------------ | --------------------------------- | ------------------------------------------------------------------------------ |
| A/B Test Control vs Target, A/B test Promo A vs Promo B| Two independent groups comparing **average metric** (revenue, time spent)                  | **Independent Two-Sample t-test** | Tests whether the **means of two unrelated groups differ**.                    |
|Same patients before vs after |Same subjects measured **before and after** (e.g., spend before vs after promotion)        | **Paired t-test**                 | Tests whether the **mean difference within the same subjects** is significant. |
|A/B test of proportions | Two groups comparing **conversion rate (binary outcome)**                                  | **Two-Proportion Z-test**         | Compares whether **two population proportions differ**.                        |
|2 Brands: converted vs not converted, Hotel reviews |Comparing **frequency counts across categories** (brand conversions, ratings distribution) | **Chi-Square Test**, **Chi-Square Test of Independance**               | Tests whether **categorical distributions are independent**.                   |
| Data **not normally distributed** or contains strong outliers                            |  | **Mann–Whitney U Test**           | Non-parametric test comparing **median ranks of two independent groups**.      |
| Comparing **3 or more groups** (Promo A, B, C) on a continuous metric                     | | **ANOVA**                         | Tests whether **at least one group mean differs** among multiple groups.       |


# One Sample test for Mean

In [0]:
# sample stats insert here
n = 100                    # sample size
X_bar = 52                 # sample mean
sample_std = None          # sample std dev (needed if sigma is unknown)

# Insert Population Parameters
μ0 = 3                # Hypothesized poulation mean, None if not known
sigma = 10             # population standard deviation, None if not known

n, μ0, X_bar, sigma, sample_std = 25, 168, 172.50, None, 15.40

In [0]:
# Compute the Confidence Interval for Population Mean
if sigma is None:
    # ---- T-interval (sigma unknown) ----
    df = n - 1
    t_score = stats.t.ppf(prob, df)
    sigma_factor = sample_std / np.sqrt(n)
    t_value = (X_bar - (μ0)) /  (sample_std / np.sqrt(n))

    print(f"Sample Mean: {X_bar}")
    print(f"Sample Std Dev: {sample_std}")

    p_value, CI_lower, CI_upper = test_output2('t', t_score, t_value, sigma_factor, X_bar)
    print_stats('Mean', 't', t_score, t_value, CI_lower, CI_upper)

else:
    # ---- Z-interval (sigma known) ----
    z_score = stats.norm.ppf(prob)
    sigma_factor = sigma / np.sqrt(n)
    Z_value = (X_bar - (μ0)) /  (sigma / np.sqrt(n))

    print(f"Sample Mean: {X_bar}")
    print(f"Population Std Dev: {sigma}")

    p_value, CI_lower, CI_upper = test_output2('Z', z_score, Z_value, sigma_factor, X_bar)
    print_stats('Mean', 'Z', z_score, Z_value, CI_lower, CI_upper)


In [0]:
# Compare if Population mean is within the confidence interval
quo = 0
if test_side == 2:
    if CI_lower <= μ0 <= CI_upper:
        quo = 1
else:
    if relative_mean == 'Lower':
        if CI_upper >= μ0:
            quo = 1
    else:
        if CI_lower <= μ0:
            quo = 1
    
if quo == 1:
    print(f"Population mean {μ0} is within the confidence interval.")
    print("Accept the Null Hypothesis")
else:
    print(f"Population mean {μ0} is NOT within the confidence interval.")
    print("Reject the Null Hypothesis")

### Determine Sample Size

# Two Sample Tests for ΔMean

### Independant Samples

In [0]:
# Sample sizes
n1, n2 = 20, 20

# Sample means, provide one of the following
X1, X2 = 3.27, 2.53
# Xd = X2 - X1

# Hypothesized Population Means, provide one of the following
μ1, μ2 = 0, 0
# μd = μ2 - μ1

# Population Variances known
sigma1, sigma2 = 1, 1

Z_score = stats.norm.ppf(prob)
sigma_factor = np.sqrt(sigma1**2/n1 + sigma2**2/n2)
Z_value = (X2 - X1 - (μ2 - μ1)) / sigma_factor

#     if relative_mean == "Lower":  # HA: μ2 - μ1 < 0
#     else:                         # HA: μ2 - μ1 > 0

p_value, CI_lower, CI_upper = test_output2('Z', Z_score, Z_value, sigma_factor, X2 - X1)
print_stats('Difference in Means', 'Z', Z_score, Z_value, CI_lower, CI_upper)


In [0]:
# Sample sizes
n1, n2 = 21, 25

# Sample means, provide one of the following
X1, X2 = 3.27, 2.53
# Xd = X2 - X1

# Hypothesized Population Means, provide one of the following
μ1, μ2 = 0, 0
# μd = μ2 - μ1

# Sample variances for Population Variances not known and assumed equal
s1, s2 = 1.30, 1.16
sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))

# Degrees of Freedom
df = n1 + n2 - 2

t_score = stats.t.ppf(prob, df)
sigma_factor = np.sqrt(sp**2/n1 + sp**2/n2)
t_value = (X2 - X1 - (μ2 - μ1)) / sigma_factor

#     if relative_mean == "Lower":   # H₁: μ₂ − μ₁ < 0
#     else:                          # H₁: μ₂ − μ₁ > 0

p_value, CI_lower, CI_upper = test_output2('t', t_score, t_value, sigma_factor, X2 - X1)
print_stats('Difference in Means', 't', t_score, t_value, CI_lower, CI_upper)

In [0]:
# Sample sizes
n1, n2 = 20, 20

# Sample means, provide one of the following
X1, X2 = 3.27, 2.53
# Xd = X2 - X1

# Hypothesized Population Means, provide one of the following
μ1, μ2 = 0, 0
# μd = μ2 - μ1

# Sample variances for Population Variances not known and not assumed equal
s1, s2 = 1.30, 1.16

# Degrees of Freedom
# Welch's T test
df = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1 - 1) + (s2**2/n2)**2/(n2 - 1))

t_score = stats.t.ppf(prob, df)
sigma_factor = np.sqrt(s1**2/n1 + s2**2/n2)
t_value = (X2 - X1 - (μ2 - μ1)) / sigma_factor

#     if relative_mean == "Lower":   # H₁: μ₂ − μ₁ < 0
#     else:                          # H₁: μ₂ − μ₁ > 0

p_value, CI_lower, CI_upper = test_output2('t', t_score, t_value, sigma_factor, X2 - X1)
print_stats('Difference in Means', 't', t_score, t_value, CI_lower, CI_upper)

### Related Sample



---

**Choosing Between Independent and Related Samples**

The choice depends entirely on the **experimental design**:

* **Use Independent Samples** when comparing **two separate populations**.
  *Example:*
  *“Do people in New York spend more than people in London?”*

* **Use Related (Paired) Samples** when measuring an **effect, change, or difference within the same group** over time or conditions.
  *Example:*
  *“Did this training program increase employee speed?”*

---

**Key Differences**

| Feature          | Independent Samples                 | Related (Paired) Samples          |
| ---------------- | ----------------------------------- | --------------------------------- |
| Subjects         | Different individuals in each group | Same individuals or matched pairs |
| Relationship     | No natural pairing                  | Observations are linked           |
| Sample Size      | Groups may differ in size           | Groups must be equal in size      |
| What is compared | Means of two groups                 | Mean of within-pair differences   |



In [0]:
# sample stats insert here
n = 5                     # paired sample size
D_bar = -4.2              # sample mean differences
sample_std = 5.67         # sample std dev of differences (needed if sigma is unknown)

# Insert Population Parameters
μ0 = 0                  # Hypothesized poulation mean difference, None if not known
sigma = None            # population standard deviation of differences, None if not known

In [0]:
# Compute the Confidence Interval for Population Mean
if sigma is None:
    # ---- T-interval (sigma unknown) ----
    df = n - 1
    t_score = stats.t.ppf(prob, df)
    sigma_factor = sample_std / np.sqrt(n)
    t_value = (D_bar - (μ0)) /  (sample_std / np.sqrt(n))

    print(f"Sample Mean: {D_bar}")
    print(f"Sample Std Dev: {sample_std}")

    p_value, CI_lower, CI_upper = test_output2('t', t_score, t_value, sigma_factor, D_bar)
    print_stats('Mean', 't', t_score, t_value, CI_lower, CI_upper)

else:
    # ---- Z-interval (sigma known) ----
    z_score = stats.norm.ppf(prob)
    sigma_factor = sigma / np.sqrt(n)
    Z_value = (D_bar - (μ0)) /  (sigma / np.sqrt(n))

    print(f"Sample Mean: {D_bar}")
    print(f"Population Std Dev: {sigma}")

    p_value, CI_lower, CI_upper = test_output2('Z', z_score, Z_value, sigma_factor, D_bar)
    print_stats('Mean', 'Z', z_score, Z_value, CI_lower, CI_upper)


In [0]:
# Compare if Population mean is within the confidence interval
quo = 0
if test_side == 2:
    if CI_lower <= μ0 <= CI_upper:
        quo = 1
else:
    if relative_mean == 'Lower':
        if CI_upper >= μ0:
            quo = 1
    else:
        if CI_lower <= μ0:
            quo = 1
    
if quo == 1:
    print(f"Population mean {μ0} is within the confidence interval.")
    print("Accept the Null Hypothesis")
else:
    print(f"Population mean {μ0} is NOT within the confidence interval.")
    print("Reject the Null Hypothesis")


# One Sample Test for Proportion

In [0]:
# sample stats insert here
n = 500                 # sample size
p = 0.05                # sample proportion
X = None                # number of successes

π = 0.08                # hypothesized population proportion

In [0]:
if p == None and X != None:
    p = X / n
if π == None:
  π = p
  
sigma_p = np.sqrt(π * (1 - π) / n) # population std deviation

if (n*π >= 5 and n*(1-π) >= 5):
  print(
        f"Normal approximation is valid with mean = {π:.3f} "
        f"\nand standard deviation = {sigma_p:.5f}"
    )
else:
  print("Cannot be approximated by a normal distribution")

In [0]:
# Standard Error
SE = np.sqrt(π * (1 - π) / n)

# Z-score
z_score = stats.norm.ppf(prob)
z_value = (p - π) / SE

    # if relative_mean == "Lower":   # H₁: p < π
    # else:                          # H₁: p > π

p_value, CI_lower, CI_upper = test_output2('Z', z_score, z_value, SE, p)

print_stats('Proportion', 'Z', z_score, z_value, CI_lower, CI_upper)

# Two Sample Tests for ΔProportions

In [0]:
n1, n2 = 72, 50                     # sample sizes
X1, X2 = 36, 31                     # number of successes
p1, p2 = X1/n1, X2/n2               # sample proportion

π1, π2 = 0, 0                       # hypothesized population proportion

In [0]:
p_bar = (X1 + X2) / (n1 + n2)

# Standard Error
SE = np.sqrt(p_bar * (1 - p_bar) * (1 / n1 + 1 / n2))

# Z-score
z_score = stats.norm.ppf(prob)
z_value = ((p1 - p2) - (π1 - π2))/ SE

# Margin Error
ME_factor = np.sqrt((p1 * (1- p1) / n1) + (p2 * (1- p2) / n2))

#     if relative_mean == 'Lower':            # HA: π2 − π1 < 0     
#     else:                                   # HA: π2 − π1 > 0

p_value, CI_lower, CI_upper = test_output2('Z', z_score, z_value, ME_factor, p1 - p2)

print_stats('Proportion Difference', 'Z', z_score, z_value, CI_lower, CI_upper)

# Two Sample test for ΔVariance 

In [0]:
# Sample sizes
n1, n2 = 21, 25

# Sample variances for Population Variances not known and assumed equal
# H0: sigma1 = sigma2
# HA: sigma1 != sigma2
s1, s2 = 1.30, 1.16

F_lower = stats.f.ppf(1 - prob, n1 - 1, n2 - 1)
F_upper = stats.f.ppf(prob, n1 - 1, n2 - 1)
F_value = s1**2 / s2**2
print(f"F-value: {F_value:.5f}")


if test_side == 2:
    print(f"F-C.I.: ({F_lower:.5f}, {F_upper:.5f})")
    if F_value < F_lower or F_value > F_upper:
        print("Reject the Null Hypothesis: Both Variances are NOT Equal")
    else:
        print("Accept the Null Hypothesis: Both Variances are Equal")
else:
    if relative_mean == 'Lower':
        print(f"F-C.I.: ({-np.inf}, {F_lower:.5f})")
    else:
        print(f"F-C.I.: ({F_upper:.5f}, {np.inf})")



### F-Test: When is it Used?

[Link](https://chatgpt.com/s/t_6a38c503c8808191858013b15dfe79f8)


The **F-test** is used to compare variances and assess the overall significance of statistical models. It relies on the **F-distribution**, which is non-negative and right-skewed.

---

 **1. Comparing Two Population Variances**

**Purpose:** Determine whether two populations have the same variance.

**Example:** Checking if the variability in exam scores from two schools is the same.

**Null Hypothesis (H₀):**
[
\sigma_1^2 = \sigma_2^2
]

**Test Statistic:**
[
F = \frac{s_1^2}{s_2^2}
]

where:

* (s_1^2) = variance of sample 1
* (s_2^2) = variance of sample 2

---

 **2. ANOVA (Analysis of Variance)**

**Purpose:** Test whether the means of **three or more groups** are significantly different.

**Example:** Comparing the average plant growth under different fertilizers.

**Null Hypothesis (H₀):**
[
\mu_1 = \mu_2 = \cdots = \mu_k
]

**Why F-test?**

It compares:

[
F = \frac{\text{Between-group variance}}{\text{Within-group variance}}
]

A large F-value suggests that at least one group mean differs significantly.

---

 **3. Regression Model Significance**

**Purpose:** Evaluate whether a regression model explains a significant amount of variation in the dependent variable.

**Example:** In multiple regression, testing whether the predictors collectively contribute to the model.

**Null Hypothesis (H₀):**
[
\beta_1 = \beta_2 = \cdots = \beta_p = 0
]

(All slope coefficients are zero.)

**Why F-test?**

It assesses the **overall fit of the regression model**.

---

### **Key Points**

* The **F-distribution** is:

  * Non-negative ((F \geq 0))
  * Right-skewed

* **Larger F-values** provide stronger evidence against the null hypothesis.

---

### **Assumptions of the F-Test**

* Independent observations/samples
* Normally distributed populations (or residuals)
* Equal variances (homogeneity of variance) for ANOVA and variance comparison tests

---

### **Quick Summary**

| Use Case              | Null Hypothesis            | Interpretation                          |
| --------------------- | -------------------------- | --------------------------------------- |
| Compare two variances | (\sigma_1^2 = \sigma_2^2)  | Tests equality of population variances  |
| ANOVA                 | All group means are equal  | Determines if at least one mean differs |
| Regression F-test     | All slope coefficients = 0 | Tests overall model significance        |

> **Rule of Thumb:** The F-test compares a measure of **explained variation** to **unexplained variation**. A sufficiently large F-statistic indicates that the observed differences are unlikely to have occurred by chance.


# Chi - squared test for proportions

## 2 categories X 2 samples 

In [0]:
import numpy as np
from scipy import stats

# -----------------------------
# Observed Frequencies (2x2)
# -----------------------------
#            Like  Dislike
# Male        30      10
# Female      20      20

# observed = np.array([
#     [30, 10],
#     [20, 20]
# ])
observed = np.array([
    [12, 108],
    [24, 156]
])

# -----------------------------
# Chi-Square Test
# -----------------------------
chi2_stat, p_value, df, expected = stats.chi2_contingency(observed)

# -----------------------------
# Critical Value
# -----------------------------
alpha = 0.05
chi2_critical = stats.chi2.ppf(1 - alpha, df)

# -----------------------------
# Output
# -----------------------------
print("Observed Frequencies:")
print(observed)

print("\nExpected Frequencies:")
print(expected.round(2))

print(f"\nChi-Square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom: {df}")
print(f"Critical Value (α = {alpha}): {chi2_critical:.4f}")
print(f"p-value: {p_value:.4f}")

# -----------------------------
# Hypothesis Decision
# -----------------------------
if chi2_stat > chi2_critical:
    print("\nReject the Null Hypothesis")
    print("Conclusion: Variables are dependent")
else:
    print("\nFail to Reject the Null Hypothesis")
    print("Conclusion: Variables are independent")


## 2 categories X n samples

In [0]:
import numpy as np
from scipy import stats

# Observed Frequencies
#           Admin Student Faculty
# Favour      60      30      20
# Oppose      40      20      20

observed = np.array([
    [60, 30, 20],
    [40, 20, 20]
])

# Chi-Square Test
chi2_stat, p_value, df, expected = stats.chi2_contingency(observed)

# Critical Value
alpha = 0.05
chi2_critical = stats.chi2.ppf(1 - alpha, df)

# Output
print("\nObserved Frequencies:\n", observed)

print("\nExpected Frequencies:")
print(expected.round(2))

print(f"\nChi-Square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom: {df}")

print(f"Critical Value (α = {alpha}): {chi2_critical:.4f}")
print(f"p-value: {p_value:.4f}")

# Decision
if chi2_stat > chi2_critical:
    print("\nReject the Null Hypothesis")
    print("Conclusion: Opinion and Group are dependent")
else:
    print("\nFail to Reject the Null Hypothesis")
    print("Conclusion: Opinion and Group are independent")

### Marascuilo Procedure (If Chi-Square is Significant)

Suppose the Chi-square test had been significant. The Marascuilo procedure compares all pairs of proportions.

In [0]:
import numpy as np
from scipy.stats import chi2
from itertools import combinations

# Favour counts
successes = np.array([60, 30, 20])

# Total respondents
totals = np.array([100, 50, 40])

groups = ["Admin", "Student", "Faculty"]

alpha = 0.05
k = len(successes)

# Proportions
p = successes / totals

critical = chi2.ppf(1 - alpha, k - 1)

print("Marascuilo Procedure\n")

for i, j in combinations(range(k), 2):

    diff = abs(p[i] - p[j])

    critical_range = np.sqrt(
        critical *
        (
            p[i]*(1-p[i])/totals[i] +
            p[j]*(1-p[j])/totals[j]
        )
    )

    print(f"{groups[i]} vs {groups[j]}")
    print(f"Difference = {diff:.4f}")
    print(f"Critical Range = {critical_range:.4f}")

    if diff > critical_range:
        print("Significant Difference\n")
    else:
        print("Not Significant\n")

## Test for Independence n X n

In [0]:
import numpy as np
from scipy import stats

observed = np.array([
    [40, 20, 15],   # High
    [35, 15, 10],   # Medium
    [25, 15, 15]    # Low
])

chi2_stat, p_value, df, expected = stats.chi2_contingency(observed)

alpha = 0.05
chi2_critical = stats.chi2.ppf(1 - alpha, df)

print("\nObserved Frequencies:\n", observed)

print("\nExpected Frequencies:")
print(expected.round(2))

print(f"\nChi-Square Statistic: {chi2_stat:.4f}",f"Degrees of Freedom: {df}")

print(f"Critical Value: {chi2_critical:.4f}",f"p-value: {p_value:.4f}")

if chi2_stat > chi2_critical:
    print("\nReject the Null Hypothesis")
    print("Conclusion: Satisfaction and Group are dependent")
else:
    print("\nFail to Reject the Null Hypothesis")
    print("Conclusion: Satisfaction and Group are independent")

### Chi-squared test: When is it used?

| Scenario                                                            | Test                               |
| ------------------------------------------------------------------- | ---------------------------------- |
| 2 Groups × 2 Categories                                             | Chi-Square Test of Independence    |
| 2 Categories × More than 2 Groups                                   | Chi-Square + Marascuilo (post-hoc) |
| Multiple Groups × Multiple Categories                               | Chi-Square Test of Independence    |
| Significant Chi-Square and need pairwise comparisons of proportions | Marascuilo Procedure               |
| Significant multi-category table and need cell-wise investigation   | Standardized Residual Analysis     |


> Marascuilo is essentially the "ANOVA post-hoc test for proportions." Use it only after an overall Chi-Square test indicates a significant difference among multiple proportions.
